# Reproducing Simmons et al. (2026) from dandiset 001777

**Paper.** Simmons DV, ..., Surmeier DJ (2026). Astrocytes mediate the dopaminergic modulation of
tonic GABAergic signaling in substantia nigra. *Science Advances* 12, eadq8840.
[doi:10.1126/sciadv.adq8840](https://doi.org/10.1126/sciadv.adq8840). The claims file
(`claims/simmons-2026.yaml`) was written from the version of record and its supplement.

**Dandiset.** [DANDI:001777](https://dandiarchive.org/dandiset/001777/0.260521.2201), version
0.260521.2201 (doi:10.48324/dandi.001777/0.260521.2201). The paper cites no dandiset version; this
is the only published version, so it is pinned. The draft has the same 412 assets and size.

**What the dandiset holds.** 412 NWB files of SNr recordings (Clampex files converted one to one),
each sweep a `VoltageClampSeries` in pA at 20 kHz: cell-attached spiking before and after bath
drugs, and whole-cell evoked IPSCs. What each file is (virus, drugs, which file is the control) is
written only in the free-text `notes` of each file, which the helper parses. There are no
SNc recordings, no recordings at room temperature, in low glucose, with SCH39166, NPPB, or
Gq-DREADD, no gabazine-then-quinpirole recordings, and no imaging, RNAscope or voltammetry.

| Claim | Tested here |
|---|---|
| fig1c-quinpirole-reduces-gpe-ipsc | tested |
| fig1b-gabazine-blocks-gpe-ipsc | blocked: no evoked IPSCs in gabazine |
| fig1e-ppr-rises | tested |
| fig2c-quinpirole-attenuates-gpe-inhibition | tested |
| fig2b-inhibition-wanes | tested |
| fig3a-quinpirole-increases-rate | tested |
| fig3b-gabazine-increases-rate | tested |
| fig3d-gabazine-occludes-quinpirole | blocked: no such recordings |
| figs1-gabazine-effect-vs-base-rate | tested |
| fig4a-gpe-gi-dreadd-reduces-ipsc | tested |
| fig4b-gpe-gi-dreadd-no-rate-change | tested |
| figs4b-astrocyte-gq-increases-rate, figs4c-nppb-decreases-rate | blocked: no such recordings |
| figs2a-d1-antagonist-increases-rate, figs2b-quinpirole-larger-than-d1 | blocked: no SCH39166 recordings |
| fig5ab-aldh1a1-dendrites-in-snr | blocked: no images |
| fig5d-shrna-higher-rate | tested |
| fig5e-shrna-gabazine-no-change | tested |
| fig5f-aldh-inhibitors-higher-rate | tested |
| fig5g-disulfiram-blocks-gabazine | tested |
| fig5h-deab-blocks-gabazine | tested |
| fig6a-rasagiline-blocks-gabazine | tested |
| figs3-snc-gi-dreadd-slows-snc, fig6c-snc-gi-dreadd-no-snr-change | blocked: no SNc recordings (fig6c depends on figs3) |
| fig6d-gat1-enriched-aldh1a1, figs5-gat-expression-in-snc | blocked: outside data, not tested |
| fig6e-nnc711-blocks-gabazine | tested |
| figs8-low-glucose-blocks-gabazine | blocked: no such recordings |
| figs9c-vigabatrin-lowers-perceval, figs9de-vigabatrin-lowers-oxphos | blocked: no imaging |
| fig7a-d2r-in-snr-astrocytes | blocked: no RNAscope |
| fig7d-astrocyte-gi-increases-rate | tested |
| fig7e-gabazine-occludes-astrocyte-gi | tested |
| fig7f-snap5114-decreases-rate | tested |
| fig7f-snap5114-occludes-quinpirole | tested |
| fig7h-nnc711-increases-rate | tested |
| figs4d-room-temperature-gabazine-no-change, figs4e-temperature-reduces-gabazine-effect | blocked: no room-temperature recordings |
| figs6-aldh1a1-ko-da-release-unchanged | blocked: no voltammetry |

**Method common to the rate claims** (claims file, M1 to M6). Spikes are detected in each sweep by
threshold on the high-passed current (`snr.detect_spikes`; the paper used Clampfit with settings
it does not give). A cell's rate in a condition is the median of its per-episode rates under two
readings of the stable window: (a) all episodes, (b) for a drug the episodes from 6 min after the
start of the drug's file. Percent change is given under the three readings of M2. Tests are
two-sided paired t tests unless the claim names another; one-sided P and the Wilcoxon test are
shown as variants. Every rate result is also shown at detection thresholds of 6 and 12 noise
units.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import snr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 80)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = snr.list_assets()
meta = snr.survey(assets)
files = snr.build_files(meta)
cells = snr.build_cells(files)
F = files.set_index("path")
print(f"{len(assets)} assets in dandiset {snr.DANDISET} version {snr.VERSION}; metadata read in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Files by protocol (session_description):")
print(files.protocol.value_counts().to_string())
print("\nUnits:", files.unit.unique().tolist(), "conversion:", files.conversion.unique().tolist(),
      "sampling rate (Hz):", files.rate_hz.unique().tolist())
print("Electrode location:", files.location.unique().tolist())
print("Sweep length (s) by protocol:")
print(files.assign(sweep_s=files.sweep_len / snr.FS).groupby("protocol").sweep_s.agg(["min", "max"]).to_string())
print("Sweeps per file by protocol:")
print(files.groupby("protocol").n_sweeps.agg(["min", "median", "max"]).to_string())
ep = files[files.n_sweeps > 1].assign(interval=lambda d: np.round(d.last_sweep_t0 / (d.n_sweeps - 1), 3))
print("Episodic files, sweep interval (last sweep start / (n_sweeps - 1), s) by protocol:")
print(ep.groupby("protocol").interval.agg(lambda x: sorted(set(x))).to_string())
print("\nSubjects:", files.subject.nunique(), "(dandiset metadata: 90); genotypes:", files.genotype.value_counts().to_dict())
print("Sex:", files.sex.value_counts().to_dict())
print("Temperature in the notes:", files.temperature.value_counts().to_dict(), "(paper, Methods: 32 to 33 C)")
hold = files.pnotes.str.extract(r"Holding potential (-?\d+)mV")[0]
print("Holding potential in the notes (mV):", hold.value_counts(dropna=False).to_dict())

The units and the sampling rate agree with the files' own description, and the 6-s "Tonict Hold"
sweeps start every 12 s, as their `experiment_description` says.
The holding potentials and temperatures in the notes are discussed under the claims that use them.

In [ ]:
dups = files[files.duplicate_of.notna()]
print(f"Files whose first sweep is identical (first 4000 samples, sweep count and length) to an earlier file: {len(dups)}")
print(dups[["path", "duplicate_of"]].to_string(index=False))
same_sess = files.assign(sess=files.path.str.extract(r"(ses-\d{8}T\d{6})")[0]).groupby(["subject", "sess"]).size()
print("\nSession ids carried by two files ('obj-' suffix):", int((same_sess > 1).sum()))
multi = files.groupby("subject").date.nunique()
print("\nSubjects with recordings on more than one date (slice experiments are terminal):")
print(files[files.subject.isin(multi[multi > 1].index)][["subject", "date", "age", "experiment"]].drop_duplicates().to_string(index=False))

Every pair of files with the same session id and an `obj-` suffix holds the same data; each is
counted once. Three subject ids carry recordings on dates 4 to 32 days apart, with ages that
advance with the date. Brain-slice experiments end the animal, so either the ids were reused or
the dates are wrong; this is recorded as a finding and does not change the counts, which are
of cells.

In [ ]:
num = dict(zip(files.path, files.orig))
show = cells.copy()
for k in ["ctrl", "drug1", "drug2", "both", "unlabelled"]:
    show[k] = show[k].map(lambda l: ",".join(f"{num[p]:03d}" for p in l))
show["flags"] = cells["flags"].map(lambda l: f"{len(l)} file(s) not named in their own notes" if l else "")
print("Cells per experiment (the helper's grouping of the notes):")
print(cells.groupby("experiment").agg(cells=("cell", "size"), subjects=("subject", "nunique")).to_string())
print()
print(show[["experiment", "subject", "date", "family", "ctrl", "drug1", "drug2", "both", "unlabelled", "flags"]]
      .sort_values(["experiment", "subject", "date"]).to_string(index=False))

The grouping follows the notes: the control file and the file the drug was applied in, by their
original Clampex numbers. Three notes are inconsistent with their files: file 0007 of subject 8952
and file 0000 of subject 6813 (Five stim) are not named in their own notes, and the nine files
0015 to 0023 of subject 9019 carry the notes of files 0003 to 0010 of the same day (a second cell),
so which of them were recorded in quinpirole is not stated. That cell is left out of the main IPSC
analysis and shown as a variant.

The counts of cells match the paper's n for most experiments: Fig. 2C 8, Fig. 4A 4, Fig. 4B 5,
Fig. 5E 15 shRNA, Fig. 5H 5, Fig. 6A 9, Fig. 6C 8, Fig. 6E 6, Fig. 7D 10, Fig. 7E 11, Fig. 7F 19,
Fig. 7H 6. Disulfiram with gabazine has 3 cells against the paper's 5, quinpirole without an opsin
7 against 8 (Fig. 3A), and gabazine alone 49 against 45 (Fig. 3B legend).

### Spike detection

The rate claims rest on spike detection. The next cell detects spikes in every sweep of the
cell-attached files (about ten minutes) and shows, on two sweeps of each experiment, the amplitudes
of the threshold crossings in noise units: noise events sit below about 8 and action currents above
about 10, which is why 8 is the main threshold.

In [ ]:
t = time.time()
rate_exps = [e for e in cells.experiment.unique() if not e.startswith(("gpe_opto", "gpe_gi_ipsc"))]
rate_paths = sorted({p for _, r in cells[cells.experiment.isin(rate_exps)].iterrows()
                     for k in ["ctrl", "drug1", "drug2", "both"] for p in r[k]})
spikes = snr.all_spikes([F.loc[p, "asset_id"] for p in rate_paths])
import resource
print(f"spikes detected in {len(rate_paths)} files in {time.time() - t:.0f} s; "
      f"peak memory of workers {resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6:.2f} GB")

In [ ]:
import h5py, remfile
from scipy import signal as _sig
rows = []
for exp in sorted(rate_exps):
    c = cells[cells.experiment == exp].iloc[0]
    p = (c["ctrl"] or c["both"])[0]
    f = h5py.File(remfile.File(snr.asset_url(F.loc[p, "asset_id"])), "r")
    a = f["acquisition"]
    key = sorted(a)[0]
    x = a[key]["data"][: int(60 * snr.FS)].astype(float)
    y = _sig.sosfiltfilt(snr._SOS, x)
    noise = np.median(np.abs(y)) / 0.6745
    pol = spikes[F.loc[p, "asset_id"]]["polarity"]
    idx, _, _ = snr.detect_spikes(x, 4.0, pol)
    z = np.array([(pol * y[i:i + 40]).max() for i in idx]) / noise
    h = np.histogram(z, bins=[4, 6, 8, 10, 15, 30, 1e9])[0]
    rows.append(dict(experiment=exp, file=p.split("/")[1][:34], noise_pA=round(noise, 2), polarity=pol,
                     **{f"z{a}-{b}": n for a, b, n in zip([4, 6, 8, 10, 15, 30], [6, 8, 10, 15, 30, "inf"], h)}))
print("Crossings of 4 noise units in the first sweep (or first 60 s) of one control file per experiment, by peak amplitude in noise units:")
print(pd.DataFrame(rows).to_string(index=False))

In most files the crossings split into a group under 8 and a group above 10 to 15 noise units; the
rate results below are repeated at 6 and 12.

In [ ]:
def rates(reading="a", k=snr.K_THRESH, window=None, note_windows=False, exps=None, cl=None):
    cl = cells if cl is None else cl
    if exps is not None:
        cl = cl[cl.experiment.isin(exps)]
    return snr.cell_rates(cl, F, spikes, reading=reading, k=k, window=window, note_windows=note_windows)


def summarize(exps, a="ctrl", b="drug1", window=None, readings=("a", "b"), ks=snr.THRESHOLDS, cl=None, note_windows=False):
    out = []
    for r in readings:
        for k in ks:
            d = rates(r, k, window, note_windows, exps, cl)
            s = snr.paired_summary(d, a, b)
            out.append(dict(reading=r, k=k, **s))
    return pd.DataFrame(out)


def by_subject(d, a="ctrl", b="drug1"):
    g = d.dropna(subset=[a, b]).groupby("subject")
    return pd.DataFrame({"cells": g.size(), f"{a}_mean": g[a].mean(), f"{b}_mean": g[b].mean(),
                         "cells_up": g.apply(lambda x: int((x[b] > x[a]).sum()))})


all_rates = rates("a", exps=rate_exps)
print("Rates (Hz, reading a, threshold 8) by experiment: median and range of control rates")
print(all_rates.groupby("experiment").ctrl.describe()[["count", "min", "50%", "max"]].round(1).to_string())

### Timing of the light train and of the evoked currents

There is no trial or stimulus table: the light and electrical stimuli are not stored (the stimulus
series hold only a -10 mV test pulse). The timing is read from the responses. The 30-Hz train of
Fig. 2 should silence spiking from 2 to 4 s of each 6-s sweep in the Chronos cells and not in the
cells without an opsin; the evoked IPSCs should start at the same time in every cell.

In [ ]:
def psth(paths, k=snr.K_THRESH, bins=np.arange(0, 6.0001, 0.1)):
    counts, n = np.zeros(len(bins) - 1), 0
    for p in paths:
        sp = spikes[F.loc[p, "asset_id"]]
        for t in sp["spikes"][k]:
            counts += np.histogram(t, bins)[0]
            n += 1
    return counts / n / np.diff(bins), n


bins = np.arange(0, 6.0001, 0.1)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for i, exp in enumerate(["gpe_train_quinpirole", "quinpirole_no_opsin"]):
    rows = []
    for _, c in cells[cells.experiment == exp].iterrows():
        h, n = psth(c["ctrl"])
        rows.append(h)
        ax[i].plot(bins[:-1] + 0.05, h, lw=0.7)
    rows = np.array(rows)
    pre = rows[:, (bins[:-1] >= 1) & (bins[:-1] < 2)].mean(1)
    on = rows[:, (bins[:-1] >= 2) & (bins[:-1] < 2.2)].mean(1)
    post = rows[:, (bins[:-1] >= 4) & (bins[:-1] < 5)].mean(1)
    print(f"{exp}: control rate 1-2 s {np.round(pre, 1)} Hz; 2.0-2.2 s {np.round(on, 1)}; 4-5 s {np.round(post, 1)}")
    ax[i].set_title(exp)
    ax[i].axvspan(2, 4, color="0.9")
    ax[i].set_xlabel("time in sweep (s)")
ax[0].set_ylabel("rate (Hz)")
plt.tight_layout()
plt.show()
print("Plotted range:", 0, "to", round(float(max(psth(c["ctrl"])[0].max() for _, c in cells[cells.experiment.isin(["gpe_train_quinpirole", "quinpirole_no_opsin"])].iterrows())), 1), "Hz; no point outside the axes.")

In every Chronos cell the rate falls at 2.0 s and recovers by 4 s, and it does not fall in the
cells without an opsin: the train runs from 2 to 4 s, as the Fig. 2 legend implies, and the spike
times share the clock of the light. The timing of the evoked IPSCs is shown with their analysis.

## fig1c-quinpirole-reduces-gpe-ipsc

> The D2R agonist quinpirole (2 uM) reduces the amplitude of optogenetically evoked GPe IPSCs in
> SNr neurons by roughly 30%. (paired t test, P = 0.02, n = 5; expected 30 +/- 3%)

The "20 Hz Light" files hold 8 sweeps of 15 s each; a train of 0.5-ms light pulses at 20 Hz starts
at 0.55 s (read from the responses below). Amplitude: peak of the mean evoked current over all
sweeps of a condition, from the mean over the 20 ms before the first pulse (claims file). The peak
is taken within 0.5 to 20 ms after the pulse. Its sign is fixed per cell by its control: the
currents are outward in cells held at -50 to -55 mV and inward in the cell held at -70 mV (notes).

In [ ]:
ipsc_cells = cells[cells.experiment == "gpe_opto_ipsc_quinpirole"].copy()
ipsc_paths = sorted({p for _, r in ipsc_cells.iterrows() for k in ["ctrl", "drug1", "unlabelled"] for p in r[k]})
t = time.time()
ipsc_X = snr.all_sweeps([F.loc[p, "asset_id"] for p in ipsc_paths], 0.50, 0.70)
print(f"read {len(ipsc_paths)} files in {time.time() - t:.0f} s")
STIM1, STIM2 = 0.55, 0.60


def cond_X(paths):
    return np.vstack([ipsc_X[F.loc[p, "asset_id"]]["X"] for p in paths])


# the unlabelled second cell of subject 9019: its notes are copies of the first cell's (files 3-6 control,
# 7-10 quinpirole); variant: the same layout shifted to its own files, 15-18 control and 19-22 quinpirole
rows = []
for _, c in ipsc_cells.iterrows():
    if c["unlabelled"]:
        ctrl = [p for p in c["unlabelled"] if 15 <= num[p] <= 18]
        drug = [p for p in c["unlabelled"] if 19 <= num[p] <= 22]
        variant_only = True
    else:
        ctrl, drug, variant_only = c["ctrl"], c["drug1"], False
    Xc, Xd = cond_X(ctrl), cond_X(drug)
    m = Xc.mean(0)
    tt = np.arange(len(m)) / snr.FS + 0.50
    seg = np.abs(m - m[(tt >= 0.53) & (tt < 0.549)].mean())
    onset = tt[(tt > 0.5505) & (seg > 0.2 * seg[(tt > 0.5505) & (tt < 0.57)].max())][0]
    a_c, sign = snr.evoked_amplitudes(Xc, 0.50, STIM1, (-0.02, -0.001), (0.0005, 0.02))
    a_d, _ = snr.evoked_amplitudes(Xd, 0.50, STIM1, (-0.02, -0.001), (0.0005, 0.02), sign)
    # paired-pulse ratio, reading (a): second peak from the baseline before the first pulse
    def second(X, base_ref):
        mm = X.mean(0)
        i = lambda s: int(round((s - 0.50) * snr.FS))
        b = mm[i(base_ref - 0.02 if base_ref == STIM1 else STIM2 - 0.002):i(base_ref - 0.001 if base_ref == STIM1 else STIM2 - 0.0001)].mean()
        seg2 = sign * (mm[i(STIM2 + 0.0005):i(STIM2 + 0.02)] - b)
        return float(seg2.max())
    rows.append(dict(cell=c["cell"], subject=c["subject"], variant_only=variant_only, sign=sign,
                     onset_ms=round(onset * 1000, 2), n_ctrl_sweeps=len(Xc), n_quin_sweeps=len(Xd),
                     amp_ctrl=a_c, amp_quin=a_d, pct_change=(a_d - a_c) / a_c * 100,
                     ppr_a_ctrl=second(Xc, STIM1) / a_c, ppr_a_quin=second(Xd, STIM1) / a_d,
                     ppr_b_ctrl=second(Xc, STIM2) / a_c, ppr_b_quin=second(Xd, STIM2) / a_d,
                     hold=F.loc[(ctrl or drug)[0], "pnotes"].split("Holding potential ")[1][:5]))
ipsc = pd.DataFrame(rows)
print(ipsc.round(3).to_string(index=False))

In [ ]:
def ipsc_test(d, a, b):
    t = stats.ttest_rel(d[b], d[a])
    pc = (d[b] - d[a]) / d[a] * 100
    return dict(n=len(d), mean_a=d[a].mean(), mean_b=d[b].mean(), t=t.statistic, p=t.pvalue,
                p_less=stats.ttest_rel(d[b], d[a], alternative="less").pvalue,
                pct_a_mean_of_cells=pc.mean(), pct_b_of_means=(d[b].mean() - d[a].mean()) / d[a].mean() * 100,
                pct_c_median_of_cells=pc.median())


main = ipsc[~ipsc.variant_only]
res = pd.DataFrame([dict(cells="5 labelled cells", **ipsc_test(main, "amp_ctrl", "amp_quin")),
                    dict(cells="6 cells (variant: 9019 second cell, files 15-18 / 19-22)", **ipsc_test(ipsc, "amp_ctrl", "amp_quin"))])
print("IPSC amplitude (pA), control against quinpirole; paired t test two-sided (p) and one-sided (p_less):")
print(res.round(4).to_string(index=False))
print("\nOnset of the first evoked current, all cells:", ipsc.onset_ms.min(), "to", ipsc.onset_ms.max(), "ms")

**Result: reproduced.** In the 5 labelled cells the evoked IPSC falls from 265.6 to 198.5 pA in
quinpirole (paired t test P = 0.0087), a reduction of 30.1% as the mean of the cells' changes
(reading a of M2, inside 30 +/- 3), 25.3% of the group means and 27.3% as the median. With the
sixth cell under the assumed file layout, P = 0.0070 and the reduction is 26.7%. The evoked
currents start 1.2 to 2.2 ms after 0.55 s in every cell.

## fig1e-ppr-rises

> Quinpirole raises the paired-pulse ratio of GPe evoked IPSCs at a 50 ms interval (paired t test,
> P = 0.02, n = 6).

Second pulse at 0.60 s, 50 ms after the first. Readings (claims file): (a) second peak from the
baseline before the first pulse; (b) second peak from the current in the 2 ms before the second
pulse. The paper's n = 6 matches the dataset's six cells only with the second cell of 9019, whose
files are labelled by the shifted layout (an assumption, shown with the five labelled cells).

In [ ]:
rows = []
for name, d in [("5 labelled cells", main), ("6 cells, 9019 second cell as files 15-18 / 19-22", ipsc)]:
    for r in ["a", "b"]:
        a, b = f"ppr_{r}_ctrl", f"ppr_{r}_quin"
        t = stats.ttest_rel(d[b], d[a])
        rows.append(dict(cells=name, reading=r, n=len(d), ppr_ctrl=d[a].mean(), ppr_quin=d[b].mean(),
                         cells_up=int((d[b] > d[a]).sum()), t=t.statistic, p=t.pvalue,
                         p_greater=stats.ttest_rel(d[b], d[a], alternative="greater").pvalue))
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** The paired-pulse ratio rises in all 5 labelled cells, from 0.868 to
0.897 (reading a, P = 0.022) and from 0.861 to 0.889 (reading b, P = 0.013). With the sixth cell
(n = 6, the paper's n) it rises in all 6 (P = 0.010 and 0.011).

## fig2c-quinpirole-attenuates-gpe-inhibition

> A 2-s, 30-Hz optogenetic train of GPe inputs inhibits SNr spiking, and quinpirole attenuates this
> inhibition. (Wilcoxon matched-pairs signed rank test, P = 0.02, n = 8)

Percent inhibition = (basal rate - rate in 2 to 4 s) / basal rate x 100 from each condition's
PSTH over all its sweeps (claims file; code part 2). Basal: (a) 0 to 2 s; (b) 0 to 2 s and 4 to
6 s. Variant: the median over sweeps of a per-sweep percent inhibition.

In [ ]:
def inhibition(paths, k, basal):
    n_sw, cb, ct, per = 0, 0, 0, []
    for p in paths:
        for t in spikes[F.loc[p, "asset_id"]]["spikes"][k]:
            b = sum(np.sum((t >= x0) & (t < x1)) for x0, x1 in basal) / sum(x1 - x0 for x0, x1 in basal)
            tr = np.sum((t >= 2) & (t < 4)) / 2.0
            cb += b; ct += tr; n_sw += 1
            if b > 0:
                per.append((b - tr) / b * 100)
    return (cb - ct) / cb * 100, np.median(per), cb / n_sw, ct / n_sw, n_sw


tc = cells[cells.experiment == "gpe_train_quinpirole"]
res, percell = [], []
for k in snr.THRESHOLDS:
    for rd, basal in [("a", [(0, 2)]), ("b", [(0, 2), (4, 6)])]:
        rows = []
        for _, c in tc.iterrows():
            ic, icm, bc, trc, nc = inhibition(c["ctrl"], k, basal)
            iq, iqm, bq, trq, nq = inhibition(c["drug1"], k, basal)
            rows.append(dict(cell=c["cell"], subject=c["subject"], inh_ctrl=ic, inh_quin=iq, inh_ctrl_sweepmed=icm,
                             inh_quin_sweepmed=iqm, basal_ctrl=bc, train_ctrl=trc, basal_quin=bq, train_quin=trq,
                             sweeps_ctrl=nc, sweeps_quin=nq))
        d = pd.DataFrame(rows)
        if k == snr.K_THRESH:
            percell.append(d.assign(reading=rd))
        for var, a, b in [("PSTH", "inh_ctrl", "inh_quin"), ("per-sweep median", "inh_ctrl_sweepmed", "inh_quin_sweepmed")]:
            w = stats.wilcoxon(d[b], d[a])
            res.append(dict(k=k, basal=rd, measure=var, n=len(d), inh_ctrl=d[a].mean(), inh_quin=d[b].mean(),
                            cells_less_inhibited=int((d[b] < d[a]).sum()), p_wilcoxon=w.pvalue,
                            p_wilcoxon_less=stats.wilcoxon(d[b], d[a], alternative="less").pvalue))
print(pd.concat(percell).round(1).to_string(index=False))
print()
print(pd.DataFrame(res).round(4).to_string(index=False))

**Result: not reproduced.** The mean percent inhibition falls from 56.8 to 42.3 in quinpirole
(PSTH, basal 0 to 2 s), less inhibition in 5 of 8 cells, but the Wilcoxon test gives P = 0.11
under both readings of the basal window, at every threshold, and P = 0.15 for the per-sweep
variant. The direction matches the paper; the test does not reach P < 0.05 (the one-sided P is
0.055).

## fig2b-inhibition-wanes

> The GPe train initially suppresses SNr spiking strongly, and the inhibition wanes during the 2-s
> train.

From each cell's control PSTH, the rate in 2.0 to 2.5 s against 3.5 to 4.0 s (claims file).

In [ ]:
rows = []
for k in snr.THRESHOLDS:
    pooled_first = pooled_last = 0
    nsw = 0
    up = 0
    per = []
    for _, c in tc.iterrows():
        f1 = l1 = n = 0
        for p in c["ctrl"]:
            for t in spikes[F.loc[p, "asset_id"]]["spikes"][k]:
                f1 += np.sum((t >= 2.0) & (t < 2.5)); l1 += np.sum((t >= 3.5) & (t < 4.0)); n += 1
        per.append((c["cell"], f1 / n / 0.5, l1 / n / 0.5))
        up += l1 > f1
        pooled_first += f1; pooled_last += l1; nsw += n
    rows.append(dict(k=k, cells=len(tc), pooled_rate_first=pooled_first / nsw / 0.5, pooled_rate_last=pooled_last / nsw / 0.5,
                     cells_last_higher=up))
    if k == snr.K_THRESH:
        print(pd.DataFrame(per, columns=["cell", "rate_2.0-2.5", "rate_3.5-4.0"]).round(2).to_string(index=False))
print(pd.DataFrame(rows).round(2).to_string(index=False))

**Result: reproduced.** In the PSTH pooled over the 8 cells the rate in the first 0.5 s of the
train is 4.26 Hz and in the last 0.5 s 11.85 Hz (threshold 8), and the last window is higher in
8 of 8 cells, at every threshold.

## fig3a-quinpirole-increases-rate

> Quinpirole alone, with no GPe stimulation, raises the basal spike rate of SNr neurons, by about
> 50% in cell-attached recordings. (paired t test, P < 0.01, n = 8; expected 50 +/- 5%)

Which cells: the Results say the rise was seen "in these experiments" (the Fig. 2 recordings) "in
the absence of GPe axon stimulation". The dataset has two sets that fit: (i) 7 cells recorded with
the same 30 Hz light protocol in mice with no opsin (no virus in the notes), rate over the whole
sweep; (ii) the 8 Chronos cells of Fig. 2, rate over 0 to 2 s, before the train. Both are run.

In [ ]:
for name, exps, window in [("(i) no opsin, whole sweep", ["quinpirole_no_opsin"], None),
                           ("(ii) Fig. 2 Chronos cells, 0-2 s", ["gpe_train_quinpirole"], (0.0, 2.0))]:
    print(name)
    print(summarize(exps, window=window).round(4).to_string(index=False))
    d = rates("a", exps=exps, window=window)
    print(d[["cell", "ctrl", "drug1"]].round(2).to_string(index=False))
    print(by_subject(d).round(2).to_string())
    print()

**Result: not reproduced.** In the 7 cells without an opsin the rate falls in quinpirole (24.8
to 21.0 Hz, P = 0.30; reading b 20.2 Hz, P = 0.20). In the 8 Chronos cells of Fig. 2, before
the train, it rises from 20.2 to 24.9 Hz (31.8% as the mean of cells' changes) with P = 0.24
(reading b: 43.2%, P = 0.20). No paired t test gives P < 0.05, under any reading or threshold; the
increase of the Chronos cells is near the paper's 50% only under reading b of M1 with reading
a of M2 (43.2%).

## fig3b-gabazine-increases-rate

> Gabazine raises the basal spike rate of SNr neurons by roughly 25%. (P < 0.01, n = 10 in the
> Results; paired t test, n = 45 in the legend; expected 25 +/- 2.5%)

All control cells with gabazine alone (claims file). Variant: for the gap-free files whose notes
give the windows of measurement, the rate in those windows.

In [ ]:
gz = summarize(["gabazine"])
print(gz.round(4).to_string(index=False))
print("\nVariant: gap-free files measured in the windows their notes give (other cells as reading a):")
print(summarize(["gabazine"], readings=("a",), note_windows=True).round(4).to_string(index=False))
d = rates("a", exps=["gabazine"])
print("\nBy protocol family (reading a, threshold 8):")
for fam, g in d.groupby("family"):
    s = snr.paired_summary(g)
    print(fam, {k: round(v, 4) for k, v in s.items()})
print("\nCells with a higher rate in gabazine:", int((d.drug1 > d.ctrl).sum()), "of", len(d))
print(by_subject(d).round(2).to_string())

**Result: partially reproduced.** Over 49 cells gabazine raises the rate from 21.4 to 23.2 Hz
under reading b (paired t test P = 0.047, Wilcoxon P = 0.0044), and from 21.4 to 22.5 Hz under
reading a (P = 0.19, Wilcoxon P = 0.023); 31 of 49 cells are faster. The increase is 9.5%, 8.2%
and 12.0% (readings a, b, c of M2, reading b of M1), against the paper's 25 +/- 2.5%. At a
threshold of 12 no paired t test gives P < 0.05. The effect is present but at about a third to a half
of the paper's size, and its significance depends on the reading and the threshold.

## figs1-gabazine-effect-vs-base-rate

> The increase in SNr spike rate with gabazine is weakly and positively correlated with the basal
> rate. (R2 = 0.11, n = 39; expected 0.11 +/- 0.011)

OLS of the change in rate (Hz) on the basal rate over the cells of fig3b (claims file).

In [ ]:
rows = []
for r in ["a", "b"]:
    for k in snr.THRESHOLDS:
        d = rates(r, k, exps=["gabazine"]).dropna(subset=["ctrl", "drug1"])
        fit = stats.linregress(d.ctrl, d.drug1 - d.ctrl)
        rows.append(dict(reading=r, k=k, n=len(d), slope=fit.slope, intercept=fit.intercept, r2=fit.rvalue ** 2, p=fit.pvalue,
                         base_range=f"{d.ctrl.min():.1f}-{d.ctrl.max():.1f}", delta_range=f"{(d.drug1 - d.ctrl).min():.1f}-{(d.drug1 - d.ctrl).max():.1f}"))
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: not reproduced.** Over 49 cells the change with gabazine is not related to the basal
rate: R2 = 0.004 with slope -0.033 (reading a) and R2 = 0.004 with slope 0.037 (reading b),
against the paper's R2 = 0.11 and a positive slope; R2 is at most 0.009 at any threshold.

## fig4a-gpe-gi-dreadd-reduces-ipsc

> Activating Gi-DREADD in GPe with CNO reduces the amplitude of electrically evoked IPSCs in SNr
> neurons, similar in size to the quinpirole effect on optogenetically evoked IPSCs. (P = 0.02 in
> the Results, paired t test P < 0.01 in the legend, n = 4)

The "Five stim" files hold 1-s sweeps with one electrical pulse at 0.40 s (artifact at 0.4002 s).
Amplitude as for fig1c: peak of the mean current within 3 to 20 ms after the pulse (after the
artifact), from the mean over 0.38 to 0.399 s; sign fixed per cell by its control.

In [ ]:
ec = cells[cells.experiment == "gpe_gi_ipsc_cno"]
paths = sorted({p for _, r in ec.iterrows() for k in ["ctrl", "drug1"] for p in r[k]})
eX = snr.all_sweeps([F.loc[p, "asset_id"] for p in paths], 0.37, 0.45)
rows = []
for _, c in ec.iterrows():
    Xc = np.vstack([eX[F.loc[p, "asset_id"]]["X"] for p in c["ctrl"]])
    Xd = np.vstack([eX[F.loc[p, "asset_id"]]["X"] for p in c["drug1"]])
    a_c, sign = snr.evoked_amplitudes(Xc, 0.37, 0.40, (-0.02, -0.001), (0.003, 0.02))
    a_d, _ = snr.evoked_amplitudes(Xd, 0.37, 0.40, (-0.02, -0.001), (0.003, 0.02), sign)
    m = Xc.mean(0) - Xc.mean(0)[:int(0.02 * snr.FS)].mean()
    tt = np.arange(len(m)) / snr.FS + 0.37
    w = (tt > 0.403) & (tt < 0.42)
    med_c, _ = snr.evoked_amplitudes(np.median(Xc, 0)[None, :], 0.37, 0.40, (-0.02, -0.001), (0.003, 0.02), sign)
    med_d, _ = snr.evoked_amplitudes(np.median(Xd, 0)[None, :], 0.37, 0.40, (-0.02, -0.001), (0.003, 0.02), sign)
    # sweeps with a spontaneous action current (a deflection over 300 pA from the sweep median) in 0.38-0.45 s,
    # leaving out the stimulus artifact (0.3995-0.403 s)
    keep = np.r_[int(0.010 * snr.FS):int(0.0295 * snr.FS), int(0.033 * snr.FS):int(0.08 * snr.FS)]
    big = lambda X: int((np.abs(X[:, keep] - np.median(X, 1, keepdims=True)).max(1) > 300).sum())
    rows.append(dict(cell=c["cell"], subject=c["subject"], sign=sign, peak_ms=round(1000 * tt[w][np.argmax(sign * m[w])], 2),
                     sweeps_ctrl=len(Xc), sweeps_cno=len(Xd), spont_ctrl=big(Xc), spont_cno=big(Xd),
                     amp_ctrl=a_c, amp_cno=a_d, pct_change=(a_d - a_c) / a_c * 100,
                     med_amp_ctrl=med_c, med_amp_cno=med_d, med_pct_change=(med_d - med_c) / med_c * 100,
                     hold=F.loc[c["ctrl"][0], "pnotes"].split("Holding potential ")[1][:5]))
e4 = pd.DataFrame(rows)
print(e4.round(3).to_string(index=False))
r4 = ipsc_test(e4, "amp_ctrl", "amp_cno")
print("\nMean of sweeps:", {k: round(float(v), 4) for k, v in r4.items()})
r4m = ipsc_test(e4, "med_amp_ctrl", "med_amp_cno")
print("Variant, median of sweeps:", {k: round(float(v), 4) for k, v in r4m.items()})
r1 = ipsc_test(main, "amp_ctrl", "amp_quin")
print("\nReduction (percent, readings a, b, c of M2): CNO, median variant", [round(-r4m[k], 1) for k in ["pct_a_mean_of_cells", "pct_b_of_means", "pct_c_median_of_cells"]], "; CNO", [round(-r4[k], 1) for k in ["pct_a_mean_of_cells", "pct_b_of_means", "pct_c_median_of_cells"]],
      "; quinpirole (fig1c, 5 cells)", [round(-r1[k], 1) for k in ["pct_a_mean_of_cells", "pct_b_of_means", "pct_c_median_of_cells"]])

**Result: not reproduced.** With the mean of sweeps (the claims file's measure) the amplitude
rises in 2 of the 4 cells, where 23 and 18 CNO sweeps carry spontaneous action currents in the
window, and the paired t test gives P = 0.42. With the median of sweeps, a variant robust to
those currents, the amplitude falls in all 4 cells, by 42.5% on average, but P = 0.13 (one-sided
0.065), and the reduction is 12.4 points from the 30.1% of fig1c, outside the 10-point margin.

## fig4b-gpe-gi-dreadd-no-rate-change

> Inhibiting GPe terminals with Gi-DREADD does not change the basal spike rate of SNr neurons. (P = 0.19, n = 5; legend paired t test P = 0.16)

A claim of absence (claims file, M6): rate before and after (CNO in slices with GPe hM4D(Gi)), two-sided paired t test at
P < 0.05; the Wilcoxon signed-rank test and one-sided P are variants. 

In [ ]:
print(summarize(['gpe_gi_rate_cno']).round(4).to_string(index=False))
d = rates("a", exps=['gpe_gi_rate_cno'])
print(d[["cell", "ctrl", "drug1"]].round(2).to_string(index=False))
print(by_subject(d).round(2).to_string())

**Result: reproduced.** In the 5 cells CNO does not change the rate significantly: 21.9 to 15.1
Hz, paired t test P = 0.27 (reading b P = 0.23). The rate falls in 3 of 5 cells; one cell drops
from 55.3 to 27.5 Hz. At a threshold of 12, reading b gives P = 0.036 (a decrease), so the
absence rests on the detection threshold.

## fig5d-shrna-higher-rate

> SNr neurons from mice with ALDH1A1 knocked down in the SNc by shRNA fire faster than control SNr
> neurons. (Welch's t test, P < 0.001, n = 18 control / 15 shRNA)

shRNA: the control-condition rate of the 15 shRNA cells. Control: the dataset labels no cell as
the control of the shRNA or Fig. 5F experiments (readings a and b of the criterion cannot be
identified). The untreated cells it holds are the 49 gabazine-alone cells, which are run as:
(i) all 49; (ii) the 32 recorded with the same episodic protocol as the shRNA cells; (iii) the 6
recorded in November 2022, the months of the shRNA recordings.

In [ ]:
def control_sets(r):
    g = rates(r, exps=["gabazine"])
    g = g.dropna(subset=["ctrl"])
    return {"(i) all gabazine-alone cells": g,
            "(ii) episodic gabazine-alone cells": g[g.family == "episodic"],
            "(iii) gabazine-alone cells, Nov 2022": g[g.date.str.startswith("2022-11")]}


def group_test(treated, label):
    rows = []
    for r in ["a", "b"]:
        for k in snr.THRESHOLDS:
            tr = rates(r, k, exps=treated)
            sets = control_sets(r) if k == snr.K_THRESH else {kk: v for kk, v in
                   {"(i) all gabazine-alone cells": rates(r, k, exps=["gabazine"]).dropna(subset=["ctrl"])}.items()}
            for name, g in sets.items():
                x = treated_rates(tr)
                y = g.ctrl.to_numpy()
                w = stats.ttest_ind(x, y, equal_var=False)
                rows.append(dict(reading=r, k=k, control=name, n_treated=len(x), n_control=len(y), mean_treated=x.mean(),
                                 mean_control=y.mean(), t=w.statistic, p_welch=w.pvalue,
                                 p_welch_greater=stats.ttest_ind(x, y, equal_var=False, alternative="greater").pvalue))
    return pd.DataFrame(rows)


def treated_rates(tr):
    # the cell's rate in the treatment: the control condition of cells pretreated with the drug, the
    # drug condition of cells to which it was applied during the recording (disulfiram_acute)
    return np.concatenate([tr[tr.experiment != "disulfiram_acute"].ctrl.dropna().to_numpy(),
                           tr[tr.experiment == "disulfiram_acute"].drug1.dropna().to_numpy()])


print(group_test(["shrna_gabazine"], "shRNA").round(5).to_string(index=False))

**Result: reproduced (resting on an assumption).** The 15 shRNA cells fire at 26.4 Hz. Against
the 31 untreated cells recorded with the same episodic protocol (20.3 Hz) Welch's test gives
P = 0.029; against the 6 recorded in November 2022 (17.2 Hz) P = 0.035; against all 49
untreated cells (21.4 Hz) P = 0.064. With all 49, P = 0.038 at a threshold of 6 and 0.14 at 12.
The paper's P < 0.001 is not approached under any reading.

## fig5e-shrna-gabazine-no-change

> In SNr neurons from ALDH1A1 shRNA mice, gabazine does not significantly change the basal spike rate. (paired t test, P = 0.05, n = 15)

A claim of absence (claims file, M6): rate before and after (gabazine in shRNA mice), two-sided paired t test at
P < 0.05; the Wilcoxon signed-rank test and one-sided P are variants. The paper's own P (0.05) sits at the threshold.

In [ ]:
print(summarize(['shrna_gabazine']).round(4).to_string(index=False))
d = rates("a", exps=['shrna_gabazine'])
print(d[["cell", "ctrl", "drug1"]].round(2).to_string(index=False))
print(by_subject(d).round(2).to_string())

**Result: reproduced.** In the 15 shRNA cells gabazine leaves the rate unchanged: 26.4 to 25.7 Hz,
paired t test P = 0.78 (reading b P = 0.78), a change of -2.5% (mean of cells), and P >= 0.63 at
every threshold.

## fig5f-aldh-inhibitors-higher-rate

> SNr neurons in slices pretreated with the aldehyde dehydrogenase inhibitors DEAB or disulfiram fire
> faster than control neurons. (Welch's t test, P < 0.01, n = 18 control / 18 inhibitor-treated)

Treated: every cell recorded in DEAB or disulfiram (claims file): the baseline of the DEAB-only
cells (8), of the DEAB-then-gabazine cells (5) and of the disulfiram-then-gabazine cells (3), and
the rate after disulfiram of the two cells where it was applied during the recording (reading b
of M1 for those two takes the episodes from 6 min). That is 18 cells, the paper's n. Control: as
for fig5d.

In [ ]:
print(group_test(["deab_baseline", "deab_gabazine", "disulfiram_gabazine", "disulfiram_acute"], "inhibitor").round(5).to_string(index=False))
tr = rates("a", exps=["deab_baseline", "deab_gabazine", "disulfiram_gabazine", "disulfiram_acute"])
print("\nTreated cells by experiment (reading a):")
print(tr.assign(rate=np.where(tr.experiment == "disulfiram_acute", tr.drug1, tr.ctrl)).groupby("experiment").rate.describe()[["count", "mean", "min", "max"]].round(2).to_string())

**Result: not reproduced.** The 18 cells in DEAB or disulfiram fire at 20.3 Hz, no faster than
the untreated cells: 21.4 Hz for all 49 (Welch P = 0.59), 20.3 Hz for the 31 episodic ones
(P = 0.99) and 17.2 Hz for the 6 of November 2022 (P = 0.39). No reading or threshold gives a
faster treated group with P < 0.05.

## fig5g-disulfiram-blocks-gabazine

> In slices treated with disulfiram (10 uM), gabazine no longer changes SNr spike rate. (paired t test, P = 0.40, n = 5)

A claim of absence (claims file, M6): rate before and after (gabazine in disulfiram), two-sided paired t test at
P < 0.05; the Wilcoxon signed-rank test and one-sided P are variants. The dataset holds 3 such cells against the paper's 5.

In [ ]:
print(summarize(['disulfiram_gabazine']).round(4).to_string(index=False))
d = rates("a", exps=['disulfiram_gabazine'])
print(d[["cell", "ctrl", "drug1"]].round(2).to_string(index=False))
print(by_subject(d).round(2).to_string())

**Result: reproduced.** In the 3 disulfiram cells the dataset holds (the paper has 5), gabazine
gives no significant change: 18.8 to 10.5 Hz, paired t test P = 0.27 (reading b P = 0.19). The
rate falls in all 3 cells, in one to 0 Hz; with 3 cells the test has little power.

## fig5h-deab-blocks-gabazine

> In slices treated with DEAB (10 uM), gabazine does not change SNr spike rate. (paired t test, P = 0.10, n = 5)

A claim of absence (claims file, M6): rate before and after (gabazine in DEAB), two-sided paired t test at
P < 0.05; the Wilcoxon signed-rank test and one-sided P are variants. 

In [ ]:
print(summarize(['deab_gabazine']).round(4).to_string(index=False))
d = rates("a", exps=['deab_gabazine'])
print(d[["cell", "ctrl", "drug1"]].round(2).to_string(index=False))
print(by_subject(d).round(2).to_string())

**Result: reproduced.** In the 5 DEAB cells gabazine gives no significant change: 20.1 to 15.1
Hz, paired t test P = 0.20 (reading b P = 0.16); P >= 0.057 at every threshold. The rate falls
by 23.6% on average, driven by one cell that drops to 0.5 Hz.

## fig6a-rasagiline-blocks-gabazine

> In slices incubated in the MAO-B inhibitor rasagiline (10 uM), gabazine has no discernible effect on SNr spike rate. (paired t test, P = 0.23, n = 9)

A claim of absence (claims file, M6): rate before and after (gabazine in rasagiline), two-sided paired t test at
P < 0.05; the Wilcoxon signed-rank test and one-sided P are variants. These files use the '30 Hz Light' protocol in mice with no opsin; the rate is taken over the whole sweep.

In [ ]:
print(summarize(['rasagiline_gabazine']).round(4).to_string(index=False))
d = rates("a", exps=['rasagiline_gabazine'])
print(d[["cell", "ctrl", "drug1"]].round(2).to_string(index=False))
print(by_subject(d).round(2).to_string())

**Result: reproduced.** In the 9 rasagiline cells gabazine gives no significant change: 17.0 to
15.5 Hz, paired t test P = 0.15 (reading b P = 0.45); P >= 0.086 at every threshold.

## fig6e-nnc711-blocks-gabazine

> In slices incubated in the GAT-1 antagonist NNC-711, gabazine fails to increase SNr spike rate. (paired t test, P = 0.32, n = 6)

A claim of absence (claims file, M6): rate before and after (gabazine in NNC-711), two-sided paired t test at
P < 0.05; the Wilcoxon signed-rank test and one-sided P are variants. 

In [ ]:
print(summarize(['nnc711_gabazine']).round(4).to_string(index=False))
d = rates("a", exps=['nnc711_gabazine'])
print(d[["cell", "ctrl", "drug1"]].round(2).to_string(index=False))
print(by_subject(d).round(2).to_string())

**Result: reproduced.** In the 6 NNC-711 cells gabazine gives no significant change: 18.7 to
17.8 Hz, paired t test P = 0.78 under both readings and every threshold.

## fig7d-astrocyte-gi-increases-rate

> Activating Gi-DREADD in SNr astrocytes with CNO raises SNr spike rate, by 28% on average. (P < 0.01, n = 10; expected 28 +/- 2.8%, readings a or b of M2)

Rate before and after (CNO, GFAP-hM4Di), claims file M1, M4; two-sided paired t test (the main reading
where the test is not named), Wilcoxon and one-sided P as variants; percent change under the
readings of M2. 

In [ ]:
print(summarize(['astro_gi_cno'], a='ctrl', b='drug1').round(4).to_string(index=False))
d = rates("a", exps=['astro_gi_cno'])
print(d[["cell", 'ctrl', 'drug1']].round(2).to_string(index=False))
print("cells up:", int((d['drug1'] > d['ctrl']).sum()), "of", int(d[['ctrl', 'drug1']].dropna().shape[0]))
print(by_subject(d, 'ctrl', 'drug1').round(2).to_string())

**Result: reproduced (resting on an assumption).** In the 10 GFAP-hM4Di cells CNO raises the
rate in 7: under reading b of M1, 14.4 to 18.6 Hz, paired t test P = 0.029, a mean increase of
30.2% (mean of cells) and 29.3% (of means), inside 28 +/- 2.8. Under reading a the increase is
25.2% and 24.5%, inside the tolerance, but P = 0.057. At a threshold of 6, reading b gives P =
0.057 and 46.2%, so the outcome rests on the threshold and the window.

## fig7e-gabazine-occludes-astrocyte-gi

> When gabazine is applied first, CNO activation of astrocytic Gi-DREADD no longer increases SNr spike rate. (P = 0.10, n = 11)

A claim of absence (claims file, M6): rate before and after (CNO in gabazine, GFAP-hM4Di), two-sided paired t test at
P < 0.05; the Wilcoxon signed-rank test and one-sided P are variants. 

In [ ]:
print(summarize(['astro_gi_gabazine_cno']).round(4).to_string(index=False))
d = rates("a", exps=['astro_gi_gabazine_cno'])
print(d[["cell", "ctrl", "drug1"]].round(2).to_string(index=False))
print(by_subject(d).round(2).to_string())

**Result: reproduced.** In the 11 cells in gabazine CNO gives no significant change: 20.3 to 21.0
Hz, paired t test P = 0.33 (reading b P = 0.84), and P >= 0.27 for the Wilcoxon variant.

## fig7f-snap5114-decreases-rate

> The GAT-3 antagonist SNAP-5114 (20 uM) lowers SNr spike rate, by 23%. (paired t test, P = 0.02, n = 19; expected 23 +/- 2.3%)

Rate before and after (SNAP-5114; 10 gap-free and 9 episodic cells), claims file M1, M4; two-sided paired t test (the main reading
where the test is not named), Wilcoxon and one-sided P as variants; percent change under the
readings of M2. Most of these recordings are gap-free; they are cut into 6-s episodes.

In [ ]:
print(summarize(['snap5114', 'snap5114_quinpirole'], a='ctrl', b='drug1').round(4).to_string(index=False))
d = rates("a", exps=['snap5114', 'snap5114_quinpirole'])
print(d[["cell", 'ctrl', 'drug1']].round(2).to_string(index=False))
print("cells down:", int((d['drug1'] < d['ctrl']).sum()), "of", int(d[['ctrl', 'drug1']].dropna().shape[0]))
print(by_subject(d, 'ctrl', 'drug1').round(2).to_string())

**Result: partially reproduced.** SNAP-5114 lowers the rate in 15 of 19 cells, from 29.6 to 26.1
Hz (paired t test P = 0.0050; reading b P = 0.0039). The decrease is 11.7%, 11.8% and 8.3%
(readings a, b, c of M2; 13.9%, 13.6% and 11.7% under reading b of M1), about half the paper's
23 +/- 2.3%, at every threshold.

## fig7f-snap5114-occludes-quinpirole

> In the presence of SNAP-5114, quinpirole has no effect on SNr spike rate. (Welch's t test, P = 0.27)

The cells of fig7f-snap5114-decreases-rate that received quinpirole after SNAP-5114 (10). Rates in
SNAP-5114 and in SNAP-5114 with quinpirole, compared as unpaired groups by a two-sided Welch's t
test (the paper's test as written); a paired t test is a variant (claims file).

In [ ]:
rows = []
for r in ["a", "b"]:
    for k in snr.THRESHOLDS:
        d = rates(r, k, exps=["snap5114_quinpirole"]).dropna(subset=["drug1", "drug2"])
        w = stats.ttest_ind(d.drug2, d.drug1, equal_var=False)
        p = stats.ttest_rel(d.drug2, d.drug1)
        rows.append(dict(reading=r, k=k, n=len(d), mean_snap=d.drug1.mean(), mean_snap_quin=d.drug2.mean(),
                         pct_of_means=(d.drug2.mean() - d.drug1.mean()) / d.drug1.mean() * 100,
                         cells_up=int((d.drug2 > d.drug1).sum()), p_welch=w.pvalue, p_paired=p.pvalue))
print(pd.DataFrame(rows).round(4).to_string(index=False))
d = rates("a", exps=["snap5114_quinpirole"])
print(d[["cell", "ctrl", "drug1", "drug2"]].round(2).to_string(index=False))

**Result: reproduced.** In the 10 cells that received quinpirole after SNAP-5114 the rate moves
from 24.3 to 25.4 Hz, Welch's t test P = 0.85 (reading b P = 0.74), P >= 0.61 at every
threshold. The paired variant is P = 0.56 here and P = 0.010 at a threshold of 12 under reading b,
an increase of 14.4%.

## fig7h-nnc711-increases-rate

> The GAT-1 antagonist NNC-711 alone raises SNr spike rate, by 36%. (P < 0.01, n = 6; expected 36 +/- 3.6%, readings a or b of M2)

Rate before and after (NNC-711 10 uM), claims file M1, M4; two-sided paired t test (the main reading
where the test is not named), Wilcoxon and one-sided P as variants; percent change under the
readings of M2. 

In [ ]:
print(summarize(['nnc711'], a='ctrl', b='drug1').round(4).to_string(index=False))
d = rates("a", exps=['nnc711'])
print(d[["cell", 'ctrl', 'drug1']].round(2).to_string(index=False))
print("cells up:", int((d['drug1'] > d['ctrl']).sum()), "of", int(d[['ctrl', 'drug1']].dropna().shape[0]))
print(by_subject(d, 'ctrl', 'drug1').round(2).to_string())

**Result: reproduced.** NNC-711 raises the rate in 6 of 6 cells, from 25.4 to 33.6 Hz under
reading b of M1 (paired t test P = 0.0041), a mean increase of 36.0% (mean of cells), inside
36 +/- 3.6. Under reading a it is 29.3% (P = 0.011). The P values are the same at every
threshold.

## Summary

| Claim | Outcome |
|---|---|
| fig1c-quinpirole-reduces-gpe-ipsc | reproduced |
| fig1b-gabazine-blocks-gpe-ipsc | blocked (data missing) |
| fig1e-ppr-rises | reproduced |
| fig2c-quinpirole-attenuates-gpe-inhibition | not reproduced |
| fig2b-inhibition-wanes | reproduced |
| fig3a-quinpirole-increases-rate | not reproduced |
| fig3b-gabazine-increases-rate | partially reproduced (assumed) |
| fig3d-gabazine-occludes-quinpirole | blocked (data missing) |
| figs1-gabazine-effect-vs-base-rate | not reproduced |
| fig4a-gpe-gi-dreadd-reduces-ipsc | not reproduced |
| fig4b-gpe-gi-dreadd-no-rate-change | reproduced (assumed) |
| figs4b, figs4c, figs2a, figs2b, fig5ab | blocked (data missing) |
| fig5d-shrna-higher-rate | reproduced (assumed) |
| fig5e-shrna-gabazine-no-change | reproduced |
| fig5f-aldh-inhibitors-higher-rate | not reproduced |
| fig5g-disulfiram-blocks-gabazine | reproduced |
| fig5h-deab-blocks-gabazine | reproduced |
| fig6a-rasagiline-blocks-gabazine | reproduced |
| figs3, fig6c | blocked (data missing; fig6c depends on figs3) |
| fig6d, figs5 | blocked (outside data, not tested) |
| fig6e-nnc711-blocks-gabazine | reproduced |
| figs8, figs9c, figs9de, fig7a, figs4d, figs4e, figs6 | blocked (data missing) |
| fig7d-astrocyte-gi-increases-rate | reproduced (assumed) |
| fig7e-gabazine-occludes-astrocyte-gi | reproduced |
| fig7f-snap5114-decreases-rate | partially reproduced |
| fig7f-snap5114-occludes-quinpirole | reproduced |
| fig7h-nnc711-increases-rate | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** Spikes are detected by a fixed threshold on the
high-passed current, not in Clampfit with the paper's (unstated) settings; every rate result is
shown at three thresholds, and four outcomes change with it (marked assumed). The stable window
of each condition is taken by the two readings of the claims file, with the drug's application
time read as the start of the file the notes name. IPSC amplitudes are measured on the mean of
all sweeps of a condition. The control cells of Fig. 5D and 5F are not labelled in the dataset;
the untreated gabazine-alone cells stand in for them.

**What was found about the dataset.** The condition of each file (virus, drugs, control or drug
file) is given only in free-text notes; there is no table of conditions and no record of the
light or electrical stimuli. Sixteen files are deposited twice. Three sets of notes do not match
their files, among them the nine files of one IPSC cell that carry another cell's notes. The
notes give 30 C for every file (the paper: 32 to 33 C) and holding potentials of -50 to -70 mV
for the optogenetic IPSCs (the paper: -60 mV). Three subject ids have recordings on dates 4 to
32 days apart. The dataset holds no SNc recordings and no recordings with SCH39166, NPPB,
Gq-DREADD, gabazine then quinpirole, low glucose or room temperature, and none of the imaging,
RNAscope or voltammetry.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")